In [7]:
"""
=============================================================================
Project: Business Performance & Financial Operations Analytics
File Name: 01_data_cleaning.py
Directory: notebooks/
Description: Ingests raw Superstore data, cleans column headers, handles date 
             types, derives core financial metrics (Cost, Margin %), aggregates 
             to unit-month grain, and generates a simulated budget.
Inputs: data/raw/Sample - Superstore.csv
Outputs: data/cleaned/cleaned_transactions.csv
         data/cleaned/monthly_unit_performance.csv
=============================================================================
"""

import numpy as np
import pandas as pd

# 1. Load the raw dataset
df = pd.read_csv(r"D:/Financial_Ops_Analytics_Project/data/raw/Sample - Superstore.csv", encoding='windows-1252')

# 2. Clean Column Names for SQL Compatibility
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

# 3. Convert Dates to Datetime Objects
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])

# 4. Derive Core Financial Metrics at the Row Level
df['Cost'] = df['Sales'] - df['Profit']
df['Margin_Pct'] = np.where(
    df['Sales'] > 0, (df['Profit'] / df['Sales']) * 100, 0
)

df.to_csv(r"D:/Financial_Ops_Analytics_Project/data/processed/cleaned_transactions.csv", index=False)

# 5. Aggregate to Unit-Month Level for SQL Analysis
df['YearMonth'] = df['Order_Date'].dt.to_period('M').astype(str)

monthly_unit = (
    df.groupby(['Region', 'Category', 'YearMonth'])
    .agg(
        Revenue=('Sales', 'sum'),
        Cost=('Cost', 'sum'),
        Profit=('Profit', 'sum'),
        Quantity=('Quantity', 'sum'),
    )
    .reset_index()
)

monthly_unit['Margin_Pct'] = np.where(
    monthly_unit['Revenue'] > 0,
    (monthly_unit['Profit'] / monthly_unit['Revenue']) * 100,
    0,
)

# 6. Simulate a Defensible Budget
np.random.seed(42)
monthly_unit['Budget'] = monthly_unit['Revenue'] * np.random.uniform(
    0.85, 1.15, size=len(monthly_unit)
)
monthly_unit['Budget'] = monthly_unit['Budget'].round(2)
monthly_unit['Revenue'] = monthly_unit['Revenue'].round(2)
monthly_unit['Cost'] = monthly_unit['Cost'].round(2)
monthly_unit['Profit'] = monthly_unit['Profit'].round(2)
monthly_unit['Margin_Pct'] = monthly_unit['Margin_Pct'].round(2)

monthly_unit.to_csv(r"D:\Financial_Ops_Analytics_Project\data\processed\monthly_unit_performance.csv", index=False)
print('Data cleaning and aggregation complete. Files successfully saved.')

Data cleaning and aggregation complete. Files successfully saved.
